# OOF error-head diagnostics

Predicted |error| (`pred`) vs observed |error| (`oof_resid`) per protein, on residues with `mask & oof_mask`, and the outliers of Fig. 6e.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")


def set_style(base=14, family="Arial"):
    plt.rcParams.update({"font.family": family, "font.size": base,
        "axes.titlesize": base, "axes.labelsize": base, "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2, "legend.fontsize": base - 2, "svg.fonttype": "none"})


set_style(base=16)
BLUE, RED, GREEN, GREY, PURPLE = "#61AFEF", "#E06C75", "#98C379", "#999999", "#C678DD"


def style_axes(ax):
    ax.grid(alpha=.3, ls="--")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


## Config

In [ ]:
PRED_PATH = Path("../../data/analysis/Fig5/predictions_N_ensemble_10seed_sorted_centered_err.json")
MIN_RESIDUES = 5
FIG_DIR = Path("Figs")
FIG_DIR.mkdir(exist_ok=True)


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.svg", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    print("saved", name)


## Load

In [ ]:
def load(pred_path):
    with open(pred_path) as f:
        d = json.load(f)

    per_protein = {}
    n_mismatched, n_nonfinite_dropped = 0, 0
    for pdb_id, pred_list in d["pred"].items():
        pred = np.array(pred_list, dtype=np.float64)
        mask = np.array(d["mask"][pdb_id], dtype=bool)
        oof_resid = np.array(d["oof_resid"][pdb_id], dtype=np.float64)
        oof_mask = np.array(d["oof_mask"][pdb_id], dtype=bool)

        lens = {len(pred), len(mask), len(oof_resid), len(oof_mask)}
        if len(lens) > 1:
            n_mismatched += 1
        n = min(lens)

        valid = mask[:n] & oof_mask[:n]
        p, g = pred[:n][valid], oof_resid[:n][valid]

        finite = np.isfinite(p) & np.isfinite(g)
        if not finite.all():
            n_nonfinite_dropped += int((~finite).sum())
        p, g = p[finite], g[finite]

        if len(p) == 0:
            continue
        per_protein[pdb_id] = (p, g)

    if n_mismatched:
        print(f"WARNING: {n_mismatched} protein(s) had mismatched array lengths across "
              f"pred/mask/oof_resid/oof_mask -- truncated to the shortest.")
    if n_nonfinite_dropped:
        print(f"WARNING: dropped {n_nonfinite_dropped} residue(s) with non-finite "
              f"pred or oof_resid.")
    return per_protein


def _corr(x, y):
    if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0:
        return float("nan"), float("nan"), float("nan"), float("nan")
    pr, p_p = pearsonr(x, y)
    sr, p_s = spearmanr(x, y)
    return pr, p_p, sr, p_s

AX_W, AX_H = 4.6, 4.2          # plot-area size in inches, identical for every outlier figure

def fixed_axes(ax_w=AX_W, ax_h=AX_H, canvas=9.0):
    """Figure whose axes box is exactly ax_w x ax_h inches, regardless of titles/labels/legends.
    savefig(bbox_inches="tight") later crops the generous canvas to the content."""
    fig = plt.figure(figsize=(canvas, canvas))
    ax = fig.add_axes([1.5 / canvas, 1.5 / canvas, ax_w / canvas, ax_h / canvas])
    return fig, ax

In [ ]:
per_protein = load(PRED_PATH)
n_proteins = len(per_protein)
n_residues = sum(len(p) for p, _ in per_protein.values())
gt_mean_err = {pdb_id: g.mean() for pdb_id, (_, g) in per_protein.items()}
print(f"{PRED_PATH}: {n_proteins} proteins, {n_residues} valid residues")


## Per-protein means

In [ ]:
means_pred = np.array([p.mean() for p, _ in per_protein.values()])
means_gt = np.array([g.mean() for _, g in per_protein.values()])
pr3, pp3, sr3, sp3 = _corr(means_pred, means_gt)
print(f"[3] cross-protein (means, n={len(means_pred)} proteins):  "
      f"Pearson={pr3:.4f} (p={pp3:.2e})  Spearman={sr3:.4f} (p={sp3:.2e})")


## Outlier check: proteins far from the cross-protein fit

IQR rule on the residual `mean_gt - (b1 * mean_pred + b0)`; positive = under-predicted.

In [ ]:
pdb_ids = list(per_protein.keys())
view3_df = pd.DataFrame({
    "pdb_id": pdb_ids,
    "mean_pred": means_pred,
    "mean_gt": means_gt,
    "n_residues": [len(per_protein[pid][0]) for pid in pdb_ids],
})

b1, b0 = np.polyfit(means_pred, means_gt, 1)   # same fit as the view 3 plot above
view3_df["fit_pred_gt"] = b1 * view3_df.mean_pred + b0
view3_df["residual"] = view3_df.mean_gt - view3_df.fit_pred_gt


def iqr_bounds(x, k=1.5):
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


lo_r, hi_r = iqr_bounds(view3_df.residual)
under_predicted = view3_df[view3_df.residual > hi_r].sort_values("residual", ascending=False).copy()
over_predicted = view3_df[view3_df.residual < lo_r].sort_values("residual").copy()

print(f"IQR bounds on residual (mean_gt - fit): [{lo_r:.3f}, {hi_r:.3f}]")
print(f"{len(under_predicted)} under-predicted outlier(s), "
      f"{len(over_predicted)} over-predicted outlier(s) out of {len(view3_df)} proteins")

print("\nunder-predicted (actual error >> predicted, sorted by residual descending):")
display(under_predicted[["pdb_id", "n_residues", "mean_pred", "mean_gt", "residual"]])

print("\nover-predicted (predicted error >> actual, sorted by residual ascending):")
display(over_predicted[["pdb_id", "n_residues", "mean_pred", "mean_gt", "residual"]])


In [ ]:
# only >=90% sequence identity to canonical ubiquitin, plus 18831_1 (89.7%, kept anyway)
UBIQUITIN_PIDS_ALL = {
    "11505_1", "11547_1", "15410_1", "16228_1", "17439_1", "17769_1", "18611_1", "18831_1",
    "19670_1", "25972_1", "27512_1", "30693_1", "34351_1", "4493_1", "4769_1", "51725_1",
    "52529_1", "5387_1", "6466_1",
}

# Two independent visual channels: COLOR = outlier status (grey normal / red under-predicted /
# purple over-predicted), SHAPE = ubiquitin-related (diamond) vs not (circle). A point that is both
# (e.g. an over-predicted ubiquitin) is simply a purple diamond, so overlaps need no extra category.
from matplotlib.lines import Line2D

is_ubi_all = view3_df.pdb_id.isin(UBIQUITIN_PIDS_ALL)
is_under = view3_df.pdb_id.isin(under_predicted.pdb_id)
is_over = view3_df.pdb_id.isin(over_predicted.pdb_id)
status_color = np.where(is_under, RED, np.where(is_over, PURPLE, GREY))

print(f"under-predicted: {is_under.sum()} ({(is_under & is_ubi_all).sum()} ubiquitin) | "
      f"over-predicted: {is_over.sum()} ({(is_over & is_ubi_all).sum()} ubiquitin) | "
      f"ubiquitin total: {is_ubi_all.sum()}")
print("over-predicted & ubiquitin:", sorted(view3_df.pdb_id[is_over & is_ubi_all]))

fig, ax = fixed_axes() 

lim = [0, max(means_pred.max(), means_gt.max()) * 1.05]
xs = np.linspace(lim[0], lim[1], 50)
ax.plot(lim, lim, "k--", lw=1, zorder=1)

# circles: non-ubiquitin proteins (normal ones small and faint, outliers larger)
m = ~is_ubi_all & ~is_under & ~is_over
ax.scatter(view3_df.mean_gt[m], view3_df.mean_pred[m], s=22, alpha=.25,
           color=GREY, edgecolor="none", zorder=2)
m = ~is_ubi_all & (is_under | is_over)
ax.scatter(view3_df.mean_gt[m], view3_df.mean_pred[m], s=70,
           color=status_color[m.values], edgecolor="none", zorder=3)

# diamonds: ubiquitin-related proteins, filled by the same outlier-status color;
# outlier diamonds drawn last so they sit on top of the dense grey ubiquitin cluster
m = is_ubi_all & ~is_under & ~is_over
ax.scatter( view3_df.mean_gt[m], view3_df.mean_pred[m], s=70, marker="D",
           color=GREY, edgecolor="k", lw=.6, zorder=4)
m = is_ubi_all & (is_under | is_over)
ax.scatter(view3_df.mean_gt[m], view3_df.mean_pred[m], s=80, marker="D",
           color=status_color[m.values], edgecolor="k", lw=.8, zorder=5)

ax.plot(b1 * xs + b0, xs, color=GREY, lw=1.5, ls="-.", zorder=1)
ax.plot(b1 * xs + b0 + hi_r, xs, color=RED, lw=1.2, ls="--", alpha=.45, zorder=1)
ax.plot(b1 * xs + b0 + lo_r, xs, color=PURPLE, lw=1.2, ls="--", alpha=.45, zorder=1)
# legend: colors (outlier status) + shape (ubiquitin), built from proxies so overlaps stay unambiguous
handles = [
    Line2D([], [], ls="none", marker="o", ms=6, mfc=GREY, mec="none", alpha=.6, label="normal"),
    Line2D([], [], ls="none", marker="o", ms=8, mfc=RED, mec="none", label="under-predicted outlier"),
    # Line2D([], [], ls="none", marker="o", ms=8, mfc=PURPLE, mec="none", label="over-predicted outlier"),
    Line2D([], [], ls="none", marker="D", ms=7, mfc="white", mec="k", mew=.6, label="ubiquitin-related"),
]
ax.legend(handles=handles, frameon=False, loc="lower right")

ax.set_ylabel("mean SLAE-ERROR prediction per protein (ppm)")
ax.set_xlabel("mean ground-truth |error| per protein (ppm)")
ax.set_title(f"Pearson={pr3:.2f}  Spearman={sr3:.2f}\nNumber of outliers={len(under_predicted) + len(over_predicted)}"
             f"/{len(view3_df)} proteins  ({is_ubi_all.sum()} ubiquitin)")
ax.set_xlim(lim)
ax.set_ylim(lim)
plt.tight_layout()
save(fig, "outliers_cross_protein")
plt.show()
